In [ ]:
# hide
import numpy as np
import pyquist as pq
from scipy.signal import lfilter

F_S = 44100


def lpf(f_c, Q, f_s):
    """Resonant low-pass biquad (Audio EQ Cookbook). High Q gives a strong
    peak at the cutoff frequency."""
    w0 = 2 * np.pi * f_c / f_s
    c, alpha = np.cos(w0), np.sin(w0) / (2 * Q)
    b = np.array([(1 - c) / 2, 1 - c, (1 - c) / 2])
    a = np.array([1 + alpha, -2 * c, 1 - alpha])
    return b / a[0], a / a[0]


def sawtooth(f0, duration, num_harmonics=40):
    """A band-limited sawtooth: a sum of harmonics with 1/k amplitudes."""
    t = np.arange(int(duration * F_S)) / F_S
    x = sum((1 / k) * np.sin(2 * np.pi * k * f0 * t) for k in range(1, num_harmonics + 1))
    return x / np.max(np.abs(x))

In [ ]:
# The classic subtractive-synthesis sound: a resonant low-pass filter whose
# cutoff sweeps over time, applied to a bright sawtooth tone. We process the
# signal block by block, redesigning the filter with a new cutoff each block.
x = sawtooth(f0=110, duration=4.0)

print("Dry sawtooth (before filtering):")
pq.play(pq.Audio(x, F_S))

block = 512
y = np.zeros_like(x)
state = np.zeros(2)                    # biquad filter memory, carried between blocks
for i in range(0, len(x), block):
    lfo = 0.5 * (1 + np.sin(2 * np.pi * 0.4 * i / F_S))   # slow 0..1 oscillation
    cutoff = 150 * (4500 / 150) ** lfo                    # sweep 150 Hz -> 4500 Hz
    b, a = lpf(f_c=cutoff, Q=6.0, f_s=F_S)                # Q=6 gives an audible resonance
    y[i:i + block], state = lfilter(b, a, x[i:i + block], zi=state)

y = y / np.max(np.abs(y))             # normalize (resonance can boost the level)
print("Filtered sweep:")
pq.play(pq.Audio(y, F_S))